# Supply Chain Disruption Response Coordinator - Demo

This notebook demonstrates the 6-agent system.
**Mode:** Deterministic (offline)

## Setup

In [ ]:
import asyncio
import json
import sys
from pathlib import Path

In [ ]:
from supply_chain.orchestrator import SupplyChainOrchestrator
from supply_chain.contracts import AgentName

print(f'Agents: {len(list(AgentName))}')

In [ ]:
demo_log = Path('logs/demo_runs.jsonl')
demo_log.parent.mkdir(exist_ok=True)
if demo_log.exists():
    demo_log.unlink()
    print(f'Cleared old log')

orchestrator = SupplyChainOrchestrator(log_path=demo_log)
print('Orchestrator ready')

## Scenario 1: Supplier Delay

In [ ]:
print('='*80)
print('SCENARIO 1: Supplier Delay')
print('='*80)

incident_1 = {
    'incident_id': 'INC-1001',
    'disruption_type': 'supplier_delay',
    'affected_product_id': 'XYZ-001',
    'affected_warehouse_id': 'WH-NYC-01',
    'required_quantity': 100,
    'original_supplier_id': 'SUP-01',
    'reported_delay_days': 7,
    'deadline_days': 10,
    'description': 'Supplier SUP-01 delayed',
}

print('\nIncident details:')
for k, v in incident_1.items():
    if k != 'reported_at':
        print(f'  {k}: {v}')

response_1 = await orchestrator.run(incident_1)

print(f'\nDecision: {response_1.decision.value}')
print(f'Selected Option: {response_1.selected_option_id}')
print(f'Outcome: {response_1.outcome}')
print(f'Run ID: {response_1.run_id}')
print(f'Duration: {response_1.duration_ms:.1f}ms')

## Scenario 2: Route Blockage

In [ ]:
print('\n' + '='*80)
print('SCENARIO 2: Route Blockage')
print('='*80)

incident_2 = {
    'incident_id': 'INC-1002',
    'disruption_type': 'route_blockage',
    'affected_product_id': 'ABC-050',
    'affected_warehouse_id': 'WH-NYC-01',
    'required_quantity': 200,
    'destination_id': 'CUST-MIA',
    'original_route_id': 'RT-NYC-MIA-01',
    'deadline_days': 5,
    'description': 'Route RT-NYC-MIA-01 blocked',
}

print('\nIncident details:')
for k, v in incident_2.items():
    if k != 'reported_at':
        print(f'  {k}: {v}')

response_2 = await orchestrator.run(incident_2)

print(f'\nDecision: {response_2.decision.value}')
print(f'Selected Option: {response_2.selected_option_id}')
print(f'Outcome: {response_2.outcome}')
print(f'Run ID: {response_2.run_id}')
print(f'Duration: {response_2.duration_ms:.1f}ms')

## Scenario 3: Invalid Incident (Missing Data)

In [ ]:
print('\n' + '='*80)
print('SCENARIO 3: Invalid Incident')
print('='*80)

incident_3 = {
    'incident_id': 'INC-1003',
    'disruption_type': 'supplier_delay',
    'affected_product_id': 'DEF-100',
    'affected_warehouse_id': 'WH-LAX-02',
    'required_quantity': 50,
    'deadline_days': 7,
    'description': 'Missing original_supplier_id',
}

print('\nIncident details (INCOMPLETE):')
for k, v in incident_3.items():
    print(f'  {k}: {v}')

response_3 = await orchestrator.run(incident_3)

print(f'\nDecision: {response_3.decision.value}')
print(f'Outcome: {response_3.outcome}')
print(f'Error: {response_3.justification}')
print(f'Run ID: {response_3.run_id}')

## Demo Log Output

In [ ]:
print('\n' + '='*80)
print('DEMO LOG FILE: logs/demo_runs.jsonl')
print('='*80)

if demo_log.exists():
    with open(demo_log) as f:
        lines = f.readlines()
    
    print(f'\nTotal entries: {len(lines)}')
    print('\nLAST 3 LOG ENTRIES:')
    print('-'*80)
    
    for i, line in enumerate(lines[-3:], 1):
        try:
            entry = json.loads(line)
            print(f'\nEntry {len(lines) - 3 + i}:')
            print(f'  Incident: {entry.get("incident_id", "N/A")}')
            print(f'  Agent: {entry.get("agent_name", "N/A")}')
            print(f'  Action: {entry.get("action", "N/A")}')
            print(f'  Section: {entry.get("section_updated", "N/A")}')
            print(f'  Status: {entry.get("status", "N/A")}')
        except:
            print(f'\nEntry {len(lines) - 3 + i}: (parse error)')
else:
    print('Log file not found')

## Summary

Demo complete! All results written to logs/demo_runs.jsonl